# 11. Multi-world systems

A `System` gathers several worlds and the orbits that connect them. One world is the **host** that the
others orbit, one is the **star** that lights them (often the same body), and each orbiting world carries
its semi-major axis and eccentricity. From that, the system computes each world's instantaneous orbital
and spin evolution in one call.

This notebook assembles a small planetary system, reads the whole system's evolution, changes which body
is the host, adds a world, and saves the result.


In [1]:
%matplotlib inline
import numpy as np

from TidalPy.constants import G
from TidalPy.structures_x.system import System
from TidalPy.structures_x.worlds.stellar import StarWorld
from TidalPy.structures_x.configs import build_world
from TidalPy.Tides_x.classes import make_tide

AU = 1.495978707e11

def tidal_planet(config):
    "Build a world and give it a fixed-Q tide so it can dissipate."
    world = build_world(config)
    world.set_tide_model(make_tide("fixed_q", {"fixed_k": [0.3], "fixed_q": [100.0]}))
    world.set_tide_config(min_degree_l=2, max_degree_l=2, eccentricity_truncation=2, obliquity_truncation=0)
    return world

star = StarWorld("Sun", 6.957e8, 1.988e30)
star.set_effective_temperature(5772.0)

inner = tidal_planet({"schema_version": "0.2.0", "name": "Inner", "type": "terrestrial",
                      "radius_m": 6.0e6, "mass_kg": 5.0e24, "spin_frequency_rad_s": 2.0e-5,
                      "layers": {"core": {"class": "physics", "type": "iron", "layer_index": 0,
                                          "radius_outer_m": 3.0e6, "is_tidal": False},
                                 "mantle": {"class": "solidliquid", "type": "mantle_rock", "layer_index": 1,
                                            "radius_fraction": 1.0, "is_tidal": True}}})
outer = tidal_planet({"schema_version": "0.2.0", "name": "Outer", "type": "gasgiant",
                      "radius_m": 6.0e7, "mass_kg": 6.0e26, "spin_frequency_rad_s": 1.0e-4,
                      "layers": {"envelope": {"class": "gas", "type": "gas", "layer_index": 0,
                                              "radius_fraction": 1.0, "is_tidal": True}}})

system = System("Demo-System")
system.add_world(star, is_host=True, is_star=True)
system.add_world(inner, semi_major_axis=0.20 * AU, eccentricity=0.05)
system.add_world(outer, semi_major_axis=0.80 * AU, eccentricity=0.10)
for w in (inner, outer):
    system.set_stellar_semi_major_axis(w, system.get_semi_major_axis(w))
    w.set_spin_frequency(system.calc_orbital_frequency(w))   # start pseudo-synchronous

print(f"{system.num_worlds} worlds: {[w.name for w in system]}")
print(f"host = {system.host.name}, star = {system.star.name}")


3 worlds: ['Sun', 'Inner', 'Outer']
host = Sun, star = Sun


## The whole system's evolution

`calc_system_evolution` returns one record per world with its instantaneous tidal heating and orbital and spin rates. The host's own rates are zero, since it is the body the others orbit.

In [2]:
rows = system.calc_system_evolution()
print(f"{'world':8} {'heating (W)':>12} {'da/dt (m/s)':>13} {'de/dt (1/s)':>13} {'dspin/dt (1/s^2)':>17}")
for r in rows:
    name = system.worlds[r["world_index"]].name
    print(f"{name:8} {r['tidal_heating']:12.3e} {r['da_dt']:13.3e} {r['de_dt']:13.3e} {r['dspin_dt']:17.3e}")


world     heating (W)   da/dt (m/s)   de/dt (1/s)  dspin/dt (1/s^2)
Sun         0.000e+00     0.000e+00     0.000e+00         0.000e+00
Inner       5.012e+11    -3.671e-12    -4.499e-22         5.361e-21
Outer       6.119e+12    -5.976e-12    -9.028e-23         4.363e-23


## Reading and setting orbital elements

Orbital elements can be read and changed at any time. Changing an eccentricity immediately changes that world's dissipation, since tidal heating grows with eccentricity.

In [3]:
print("before:", system.get_eccentricity(inner), "->", end=" ")
system.set_eccentricity(inner, 0.10)
print(system.get_eccentricity(inner))

heating_before = [r for r in system.calc_system_evolution() if system.worlds[r["world_index"]].name == "Inner"][0]
print(f"Inner heating at e=0.10: {heating_before['tidal_heating']:.3e} W")


before: 0.05 -> 0.1
Inner heating at e=0.10: 2.005e+12 W


## Changing roles

Any world can be made the host or the star with `set_host` and `set_star`. Reassigning the host
changes what the other worlds are considered to orbit (their stored orbital elements are then
interpreted about the new host), so only do this for a physically sensible reassignment. Here we
briefly make the inner planet the host and then restore the Sun.

In [4]:
print("current host:", system.host.name, "| star:", system.star.name)
system.set_host(inner)
print("after set_host(inner):", system.host.name, "| star:", system.star.name)
system.set_host(star)
print("restored:", system.host.name, "| star:", system.star.name)

current host: Sun | star: Sun
after set_host(inner): Inner | star: Sun
restored: Sun | star: Sun


## Adding and removing worlds

More worlds can be added at any time with `add_world`. There is no in-place remove; to drop a world, rebuild the system from the ones you want to keep (systems are cheap to assemble). Here we add a third planet.

In [5]:
third = tidal_planet({"schema_version": "0.2.0", "name": "Third", "type": "terrestrial",
                      "radius_m": 4.0e6, "mass_kg": 2.0e24, "spin_frequency_rad_s": 3.0e-5,
                      "layers": {"mantle": {"class": "solidliquid", "type": "mantle_rock", "layer_index": 0,
                                            "radius_fraction": 1.0, "is_tidal": True}}})
system.add_world(third, semi_major_axis=0.50 * AU, eccentricity=0.02)
system.set_stellar_semi_major_axis(third, system.get_semi_major_axis(third))
print(f"now {system.num_worlds} worlds: {[w.name for w in system]}")


now 4 worlds: ['Sun', 'Inner', 'Outer', 'Third']


## Saving the system

A complete system saves to TOML (a readable recipe) or to a binary snapshot, as in the Basics notebook. The binary form restores each world as its concrete type.

In [6]:
import tempfile
from pathlib import Path
from TidalPy.structures_x.system import System

work = Path(tempfile.mkdtemp(prefix="tidalpy_sys_"))
system.save_binary(str(work / "system.tpyb"))
reloaded = System()
reloaded.load_binary(str(work / "system.tpyb"))
print("reloaded:", [(w.name, type(w).__name__) for w in reloaded])

import shutil
shutil.rmtree(work, ignore_errors=True)


reloaded: [('Sun', 'StarWorld'), ('Inner', 'LayeredWorld'), ('Outer', 'GasGiantWorld'), ('Third', 'LayeredWorld')]


## Recap

- `System` links a host, a star, and orbiting worlds; `add_world` sets roles and orbital elements, and
  `set_host` / `set_star` reassign them.
- `calc_system_evolution` returns the instantaneous heating and orbital and spin rates for every orbiting
  world in one call; `calc_world_evolution` does it for one world.
- Orbital elements are read and set with `get_`/`set_semi_major_axis` and `get_`/`set_eccentricity`, and
  changes take effect immediately. Systems save and load like any world.

Next: **`12_thermal_orbital_evolution`** turns these instantaneous rates into a coupled ODE and integrates
the interior and orbit together over time.
